In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

np.random.seed(42)

In [2]:
BASE_DIR = Path(
    "paper_implementation"
)

PROFILE_DIR = Path(
    "daily_profiles_24h"
)

MODE2_DIR = (
    BASE_DIR /
    "theft_simulation" /
    "mode_2"
)

MODE2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
selected_df = pd.read_csv(
    BASE_DIR /
    "selected_150_meters" /
    "selected_150_meter_ids.csv"
)

fraud_df = pd.read_csv(
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

selected_meters = (
    selected_df["Meter"]
    .astype(str)
    .tolist()
)

fraud_meters = set(
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Selected Consumers:",
    len(selected_meters)
)

print(
    "Fraud Consumers:",
    len(fraud_meters)
)

Selected Consumers: 150
Fraud Consumers: 27


In [4]:
hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [5]:
total_modified = 0

log_records = []

for meter in selected_meters:

    df = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    if meter in fraud_meters:

        for row_idx in df.index:

            date = df.loc[
                row_idx,
                "DATE"
            ]

            daily_mean = (
                df.loc[
                    row_idx,
                    hour_cols
                ]
                .astype(float)
                .mean()
            )

            for hour in hour_cols:

                alpha_t = np.random.uniform(
                    0.1,
                    0.8
                )

                x_prime = (
                    alpha_t
                    * daily_mean
                )

                log_records.append({

                    "Meter":
                    meter,

                    "Date":
                    date,

                    "Hour":
                    hour,

                    "Daily_Mean":
                    daily_mean,

                    "alpha_t":
                    alpha_t,

                    "x_prime":
                    x_prime
                })

                df.loc[
                    row_idx,
                    hour
                ] = x_prime

                total_modified += 1

    df.to_csv(
        MODE2_DIR /
        f"{meter}.csv",
        index=False
    )

log_df = pd.DataFrame(
    log_records
)

log_df.to_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_2_generation_log.csv",
    index=False
)

print(
    "Modified readings:",
    total_modified
)

print(
    "Log rows:",
    len(log_df)
)

Modified readings: 20088
Log rows: 20088


In [6]:
print(
    "Files Generated:",
    len(
        list(
            MODE2_DIR.glob("*.csv")
        )
    )
)

log_df = pd.read_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_2_generation_log.csv"
)

print(
    "Log Rows:",
    len(log_df)
)

Files Generated: 150
Log Rows: 20088


In [7]:
import numpy as np

normal_meters = [
    m for m in selected_meters
    if m not in fraud_meters
]

unchanged_count = 0

for meter in normal_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode2 = pd.read_csv(
        MODE2_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode2[hour_cols].values
    )

    if same:
        unchanged_count += 1

print(
    "Unchanged Normal Consumers:",
    unchanged_count,
    "/",
    len(normal_meters)
)

Unchanged Normal Consumers: 123 / 123


In [8]:
modified_count = 0

for meter in fraud_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode2 = pd.read_csv(
        MODE2_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode2[hour_cols].values
    )

    if not same:
        modified_count += 1

print(
    "Modified Fraud Consumers:",
    modified_count,
    "/",
    len(fraud_meters)
)

Modified Fraud Consumers: 27 / 27


In [9]:
log_df.head(20)

,Meter,Date,Hour,Daily_Mean,alpha_t,x_prime
0,6270,2018-07-01,HOUR_0,2.474725,0.362178,0.896291
1,6270,2018-07-01,HOUR_1,2.474725,0.765500,1.894402
2,6270,2018-07-01,HOUR_2,2.474725,0.612396,1.515511
3,6270,2018-07-01,HOUR_3,2.474725,0.519061,1.284533
4,6270,2018-07-01,HOUR_4,2.474725,0.209213,0.517745
5,6270,2018-07-01,HOUR_5,2.474725,0.209196,0.517703
6,6270,2018-07-01,HOUR_6,2.474725,0.140659,0.348091
7,6270,2018-07-01,HOUR_7,2.474725,0.706323,1.747956
8,6270,2018-07-01,HOUR_8,2.474725,0.520781,1.288789
9,6270,2018-07-01,HOUR_9,2.474725,0.595651,1.474072
